In [ ]:
!pip install ipywidgets

In [1]:
from pathlib import Path
import sys

def find_project_root(marker_folder="research_agent"):
    """Walks up directory tree from the current working directory until it finds the project marker."""
    current_path = Path.cwd()
    for parent in [current_path] + list(current_path.parents):
        if (parent / marker_folder).exists():
            return parent
    return current_path # Fallback to current directory if not found

# 1. Dynamically set the root directory
ROOT_DIR = find_project_root("research_agent")
print(f"Project Root set to: {ROOT_DIR}")

# 2. Automatically add it to Python's system path for clean imports
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

Project Root set to: d:\projects\research-agent


In [2]:
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from pathlib import Path
from research_agent.rag_system.dense_retriever import DenseRetriever

d:\projects\research-agent\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[RAG][EMBEDDING] Loading model at import: BAAI/bge-m3


d:\projects\research-agent\venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in D:\projects\research-agent\research_agent\rag_system\evaluate_rag_system\.models\models--BAAI--bge-m3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 8200.52it/s]


In [11]:
from pathlib import Path

persist_directory = r"D:\projects\research-agent\.chroma\paper_eval_session"
session_id = "paper_eval_session"
retriever = DenseRetriever(
    session_id=session_id,
    persist_directory=persist_directory
)
count = retriever.vector_store._collection.count()
print(f"Number of chunks: {count}")

Number of chunks: 566


In [4]:
results = retriever.search(query = "Why does GPU-CFR compile a fixed game tree into a static dataflow representation?", k=2)

In [10]:
import json
for result in results:
    print(result.model_dump())

{'chunk_id': '136b7342-88f2-45f9-b592-4f64c5876d8b', 'document': {'id': '10224864-d5d9-41f2-8a50-a755d7323a4f', 'metadata': {'author': ['Boning Li', 'Longbo Huang'], 'section_headings': ['**GPU-CFR: 80x Faster Counterfactual Regret Minimization by Compiling the Game to Static Dataflow and CUDA Graph Replay**', '**1. Introduction**'], 'title': 'GPU-CFR: 80x Faster Counterfactual Regret Minimization by Compiling the Game to Static Dataflow and CUDA Graph Replay', 'source': '01_GPU-CFR_ 80x Faster Counterfactual Regret Minimization by Compiling the Game to Static Dataflow and CUDA Graph Replay.pdf'}, 'page_content': 'GPU-CFR: 80x Faster Counterfactual Regret Minimization by Compiling the Game to Static Dataflow and CUDA Graph Replay\n**GPU-CFR: 80x Faster Counterfactual Regret Minimization by Compiling the Game to Static Dataflow and CUDA Graph Replay**\n**1. Introduction**\n\n<!-- Start of picture text -->\nOpenSpiel (Python CFR) Ours CPU (8 threads)<br>LiteEFG (C++, 1 thread) GPU-CFR (A

In [12]:
import re


def _safe_collection_name(session_id: str) -> str:
    value = re.sub(r"[^a-zA-Z0-9_-]", "_", session_id)
    value = value[:50] or "default"
    return f"research_documents_{value}"

embedding = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    cache_folder=r"D:\projects\research-agent\.models",
    model_kwargs={"device": "cpu", "local_files_only": False},
    encode_kwargs={"normalize_embeddings": True}
)

vector_store = Chroma(
    embedding_function=embedding,
    collection_name=_safe_collection_name(session_id=session_id),
    persist_directory=persist_directory,
)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 9835.63it/s]


In [13]:
count = retriever.vector_store._collection.count()
print(f"Number of chunks: {count}")

Number of chunks: 566


In [14]:
query = "Why does GPU-CFR compile a fixed game tree into a static dataflow representation?"
k = 2

In [15]:
results = vector_store.similarity_search_with_relevance_scores(query=query, k=k)

In [21]:
for (document, score) in results:
    print("id : ", document.id)
    print("page content : ", document.page_content)
    print("metadata : ", document.metadata)
    print("score : " , score,  "\n\n")

id :  10224864-d5d9-41f2-8a50-a755d7323a4f
page content :  GPU-CFR: 80x Faster Counterfactual Regret Minimization by Compiling the Game to Static Dataflow and CUDA Graph Replay
**GPU-CFR: 80x Faster Counterfactual Regret Minimization by Compiling the Game to Static Dataflow and CUDA Graph Replay**
**1. Introduction**

<!-- Start of picture text -->
OpenSpiel (Python CFR) Ours CPU (8 threads)<br>LiteEFG (C++, 1 thread) GPU-CFR (A100, eager)<br>Kim 2026 (A100, CuPy) GPU-CFR (A100, CUDA graph)<br>3<br>10<br>2<br>10<br>185× 213×<br>1<br>10<br>0<br>10<br>1<br>10<br>2<br>10<br>2 3 4 5<br>10 10 10 10<br>Game size (Infosets)<br>Time per solver iteration (ms)<br><!-- End of picture text -->  
**Figure 1** | Steady-state time per CFR iteration versus game size (both axes logarithmic; lower is better); backends and variants as in Table 3.  
To remove these obstacles, we propose GPU-CFR, which treats a fixed game as a program to be compiled. GPU-CFR walks a game once (two players, zero-sum, perfec